# Table Structure Recognition Dataset Profiler

This notebook profiles and analyzes structural properties of two table recognition datasets: **SciTSR** (scientific papers) and **Kenny** (multiple domains).

In [11]:

from pathlib import Path
import json, re
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from PIL import Image

SCITSR_ROOT   = Path("input")                     
SCITSR_SPLITS = ["train", "test"]
KENNY_ROOT    = Path("kenny_data/input_images")   
KENNY_DOMAINS = ["Biology", "CompSci", "ICDAR", "MatSci"]

SCITSR_VISION_PRED = Path("outputs/vision_agent_gemma4_SciTSR_run")   
SCITSR_TEXT_PRED   = Path("outputs/text_agent_gemma4_SciTSR_run")     
KENNY_VISION_PRED  = Path("outputs/vision_agent_gemma4_run")     
KENNY_TEXT_PRED    = Path("outputs/text_agent_gemma4_run")        

ONLY_EVALUATED = True   # True = count only tables BOTH agents predicted 
IMG_EXTS = (".png")

# ---------------------------------------------------------------------
def table_metrics(cells):
    if not cells:
        return None
    
    n_rows = max(c["er"] for c in cells) + 1
    n_cols = max(c["ec"] for c in cells) + 1

    n_multirow = sum(1 for c in cells if c["er"] > c["sr"])
    n_multicol = sum(1 for c in cells if c["ec"] > c["sc"])

    covered = sum((c["er"]-c["sr"]+1)*(c["ec"]-c["sc"]+1) for c in cells)
    density = covered / (n_rows*n_cols) if n_rows*n_cols else 0.0

    return dict(
        n_rows=n_rows, 
        n_cols=n_cols, 
        n_cells=len(cells),
        n_multirow=n_multirow, 
        n_multicol=n_multicol,
        has_multirow=n_multirow > 0, 
        has_multicol=n_multicol > 0,
        density=density
    )

def find_image(stem, *dirs):

    for d in dirs:

        if d is None or not Path(d).exists():
            continue

        for ext in IMG_EXTS:
            p = Path(d) / f"{stem}{ext}"

            if p.exists():
                return p
            
    return None

def stems_in(d, pattern="*.json"):

    return {p.stem for p in Path(d).glob(pattern)} if Path(d).exists() else set()

def parse_scitsr_json(path):

    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return []
    
    cells = data.get("cells", data) if isinstance(data, dict) else data

    out = []
    for c in cells:

        sr = c.get("start_row", c.get("sr"))
        sc = c.get("start_col", c.get("sc"))

        if sr is None or sc is None:
            continue

        er = c.get("end_row", c.get("er", sr))
        ec = c.get("end_col", c.get("ec", sc))

        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec)
        })

    return out

def parse_kenny_xml(path):

    try:
        root = ET.parse(path).getroot()
    except Exception:
        return []
    
    out = []
    for c in root.iter("cell"):

        sr = c.get("start_row")
        sc = c.get("start_col")

        if sr is None or sc is None:
            continue

        er = c.get("end_row", sr) 
        ec = c.get("end_col", sc)

        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec)
        })

    return out

# ---------------------------------------------------------------------
def profile_scitsr():
    rows = []

    for split in SCITSR_SPLITS:
        
        gt_dir  = SCITSR_ROOT / split / "structure_processed"
        img_dir = SCITSR_ROOT / split / "img"

        if not gt_dir.exists():
            continue

        used = None

        if ONLY_EVALUATED:

            vis = stems_in(SCITSR_VISION_PRED / split / "predictions")
            txt = stems_in(SCITSR_TEXT_PRED / split / "predictions")

            used = vis & txt
            print(f"[scitsr/{split}] GT={len(list(gt_dir.glob('*.json')))} "
                  f"vision_pred={len(vis)} text_pred={len(txt)} -> evaluated={len(used)}")
            
        for gt in sorted(gt_dir.glob("*.json")):

            if used is not None and gt.stem not in used:
                continue

            m = table_metrics(parse_scitsr_json(gt))

            if m is None:
                continue

            rows.append({
                "dataset": "SciTSR", 
                "domain": "Scientific (papers)", 
                "split": split,
                "table_id": f"{split}::{gt.stem}", **m,
            })
    return pd.DataFrame(rows)

def profile_kenny():
    rows = []

    for domain in KENNY_DOMAINS:

        gt_dir = KENNY_ROOT / domain / "xmls"
        if not gt_dir.exists():
            continue

        img_dirs = [KENNY_ROOT / domain / "images", KENNY_ROOT / domain / "img", KENNY_ROOT / domain]

        used = None

        if ONLY_EVALUATED:

            vis = stems_in(KENNY_VISION_PRED / domain / "predictions")
            txt = stems_in(KENNY_TEXT_PRED / domain / "nougat" / "predictions")
            used = vis & txt

            print(f"[kenny/{domain}] GT={len(list(gt_dir.glob('*.xml')))} "
                  f"vision_pred={len(vis)} text_pred={len(txt)} -> evaluated={len(used)}")
            
        for gt in sorted(gt_dir.glob("*.xml")):

            if used is not None and gt.stem not in used:
                continue

            m = table_metrics(parse_kenny_xml(gt))

            if m is None:
                continue

            rows.append({
                "dataset": "Kenny", 
                "domain": domain, 
                "split": "all",
                "table_id": f"{domain}::{gt.stem}", **m,
            })
    return pd.DataFrame(rows)

scitsr_tables = profile_scitsr()
kenny_tables  = profile_kenny()
per_table = pd.concat([scitsr_tables, kenny_tables], ignore_index=True)
print(f"\nProfiled {len(per_table)} evaluated tables "
      f"(SciTSR={len(scitsr_tables)}, Kenny={len(kenny_tables)})\n")


def summarize(df, label, domain_label):
    s = pd.Series(dtype=object)
    s["Dataset"] = label
    s["Domain"] = domain_label
    s["# Tables"] = len(df)
    s["# cells (total)"] = int(df["n_cells"].sum())
    s["Rows (mean)"] = round(df["n_rows"].mean(), 1)
    s["Rows (max)"] = int(df["n_rows"].max())
    s["Cols (mean)"] = round(df["n_cols"].mean(), 1)
    s["Cols (max)"] = int(df["n_cols"].max())
    s["% multirow"] = round(100 * df["has_multirow"].mean(), 1)
    s["% multicol"] = round(100 * df["has_multicol"].mean(), 1)
    s["Density (mean)"] = round(df["density"].mean(), 2)
    return s

summary_rows = []
if len(scitsr_tables):
    summary_rows.append(summarize(scitsr_tables, "SciTSR", "Scientific (papers)"))

if len(kenny_tables):
    dom = ", ".join(sorted(kenny_tables["domain"].unique()))
    summary_rows.append(summarize(kenny_tables, "Kenny", dom))

summary = pd.DataFrame(summary_rows)

kenny_by_domain = pd.DataFrame(
    [summarize(g, f"Kenny / {d}", d) for d, g in kenny_tables.groupby("domain")]
) if len(kenny_tables) else pd.DataFrame()

print("================ DATASET SUMMARY ================")
print(summary.to_string(index=False))

if len(kenny_by_domain):
    print("\n--------- Kenny per-domain breakdown ----------")
    print(kenny_by_domain.to_string(index=False))

[scitsr/train] GT=896 vision_pred=0 text_pred=0 -> evaluated=0
[scitsr/test] GT=666 vision_pred=157 text_pred=157 -> evaluated=157
[kenny/Biology] GT=13 vision_pred=13 text_pred=13 -> evaluated=13
[kenny/CompSci] GT=41 vision_pred=41 text_pred=41 -> evaluated=41
[kenny/ICDAR] GT=55 vision_pred=54 text_pred=55 -> evaluated=54
[kenny/MatSci] GT=47 vision_pred=47 text_pred=47 -> evaluated=47

Profiled 312 evaluated tables (SciTSR=157, Kenny=155)

================ DATASET SUMMARY ================
Dataset                          Domain  # Tables  # cells (total)  Rows (mean)  Rows (max)  Cols (mean)  Cols (max)  % multirow  % multicol  Density (mean)
 SciTSR             Scientific (papers)       157             9287          9.7          30          6.7          17        37.6        86.0            0.99
  Kenny Biology, CompSci, ICDAR, MatSci       155             9491         11.2          43          5.5          14        18.1        30.3            1.00

--------- Kenny per-domain bre